# モデル構築


## 目的・方針

- `workspace.silver` の `products` / `sales` / `inventory` から直接特徴量を作成し、店舗×商品×日 単位の「日次売上数量」を予測する回帰モデルを構築する
- Goldレイヤーの結合済みテーブル（`_30_gold_daily_sales_summary`）は使わず、モデル用notebookとして特徴量エンジニアリングも自己完結させる
- 予測対象: `quantity`（日次販売数量の合計）の回帰予測、粒度は (sale_date, store_id, product_id)
- 特徴量: 商品属性（category, unit_price）、店舗別在庫スナップショット（stock_quantity）、カレンダー特徴（day_of_week, is_weekend）
- 売上のない (store, product, date) の組み合わせも「数量0」の正当なデータ点として扱うdenseパネルを構築する（トランザクションデータの欠落ではなく、需要ゼロの実績として扱う）
- `sales_amount` は `quantity × unit_price` に由来し目的変数のリーケージとなるため特徴量には使わない（参考列としてのみ保持）
- 学習/検証は時系列分割（直近7日をテスト）とし、ランダムシャッフルは行わない
- モデルは `LightGBM`（`objective="tweedie"`）を採用する。二乗誤差（MSE）は誤差が対称・等分散であることを仮定するが、日次売上数量はゼロインフレーション・平均に応じて分散が変化するカウントデータであるため、TweedieやPoissonのような分散が平均に応じてスケールする損失関数の方が実態に即している
- MLflowでrunごとにパラメータ・メトリクス・モデルを記録し、Unity Catalog Model Registry（`workspace.model.daily_sales_quantity_predictor`）に登録する
- AutoMLやバッチ推論は本notebookのスコープ外とする

In [ ]:
!pip install mlflow lightgbm scikit-learn

In [ ]:
%restart_python

In [ ]:
from pyspark.sql.functions import *
import pandas as pd
import numpy as np

import lightgbm as lgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import mlflow
import mlflow.lightgbm
from mlflow.models.signature import infer_signature
from mlflow.tracking import MlflowClient

mlflow.set_registry_uri("databricks-uc")

In [ ]:
# UC Model Registryへの登録にはスキーマが事前に存在している必要がある
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.model")

In [ ]:
sales_df = spark.read.table("workspace.silver._20_silver_sales")
products_df = spark.read.table("workspace.silver._20_silver_products")
inventory_df = spark.read.table("workspace.silver._20_silver_inventory")

## denseパネルの構築方針

`_20_silver_sales` はトランザクション単位のデータであり、全ての(日付, 店舗, 商品)の組み合わせに行が存在するわけではない。「日次売上数量予測」というタスクの性質上、売上が発生しなかった日は「欠測」ではなく「数量0という正当な観測」として扱うべきであるため、カレンダー日付 × 店舗×商品マスタ の全組み合わせに対して実績売上を左結合し、欠損は0で埋めるdenseパネルを構築する。

この結果、多くの(店舗, 商品, 日)の組み合わせで目的変数が0となるゼロインフレーションが発生するが、これは本practiceデータの規模・粒度における既知の単純化（baselineモデルとして許容）として扱う。

In [ ]:
# 1) 売上をsale_date × store_id × product_id単位に集計
daily_sales = (
    sales_df.groupBy("sale_date", "store_id", "product_id")
    .agg(
        sum("quantity").alias("quantity"),
        sum("sales_amount").alias("sales_amount"),  # 参考列(特徴量には使わない。リーケージ注記を参照)
    )
)

# 2) カレンダー(実データのmin/max日付から動的に生成。ハードコードしない)
date_bounds = sales_df.select(
    min("sale_date").alias("min_date"),
    max("sale_date").alias("max_date"),
).first()

calendar_df = spark.sql(
    f"""
    SELECT explode(sequence(
        to_date('{date_bounds.min_date}'),
        to_date('{date_bounds.max_date}'),
        interval 1 day
    )) AS sale_date
    """
)

# 3) 店舗×商品マスタ(inventoryは店舗×商品の全組み合わせを静的に1行ずつ持つ)
store_product_df = inventory_df.select("store_id", "product_id").distinct()

# 4) denseパネル生成: カレンダー × 店舗×商品 → 実績売上を左結合 → 欠損は0埋め
panel_df = (
    calendar_df.crossJoin(store_product_df)
    .join(daily_sales, on=["sale_date", "store_id", "product_id"], how="left")
    .fillna({"quantity": 0, "sales_amount": 0})
)

In [ ]:
feature_df = (
    panel_df
    .join(products_df.select("product_id", "category", "unit_price"), on="product_id", how="left")
    .join(inventory_df.select("store_id", "product_id", "stock_quantity"), on=["store_id", "product_id"], how="left")
    .withColumn("day_of_week", dayofweek("sale_date"))  # Spark: 1=日曜, 7=土曜
    .withColumn("is_weekend", when(col("day_of_week").isin(1, 7), 1).otherwise(0))
)

## driverへのデータ集約について

本notebookの特徴量テーブルは 店舗4 × 商品16 × 日付30 = 最大1,920行 であり、scikit-learnで学習するためにdriverへ集約（`toPandas()`）してもサイズ的にリスクが小さいことを確認した上での意図的な判断である（collect/toPandasは一般的にリスクとして扱うが、本ケースは規模的に許容する）。

In [ ]:
# 意図的なdriver集約: 最大1,920行程度でサイズ的に安全と判断(上記markdown参照)
pdf = feature_df.toPandas()

for c in ["store_id", "product_id", "category"]:
    pdf[f"{c}_code"] = pdf[c].astype("category").cat.codes

pdf = pdf.sort_values("sale_date").reset_index(drop=True)

FEATURE_COLS = [
    "store_id_code",
    "product_id_code",
    "category_code",
    "unit_price",
    "stock_quantity",
    "day_of_week",
    "is_weekend",
]
TARGET_COL = "quantity"
# 注意: sales_amount は quantity × unit_price に由来し目的変数のリーケージとなるため
# 特徴量には含めない(参考列としてのみpdfに保持)

In [ ]:
TEST_DAYS = 7
max_date = pdf["sale_date"].max()
cutoff_date = max_date - pd.Timedelta(days=TEST_DAYS - 1)

train_df = pdf[pdf["sale_date"] < cutoff_date]
test_df = pdf[pdf["sale_date"] >= cutoff_date]

X_train, y_train = train_df[FEATURE_COLS], train_df[TARGET_COL]
X_test, y_test = test_df[FEATURE_COLS], test_df[TARGET_COL]

print(f"train: {X_train.shape}, test: {X_test.shape}, cutoff_date: {cutoff_date.date()}")

In [ ]:
mlflow.lightgbm.autolog(log_models=False, silent=True)  # パラメータ等は自動記録、モデルは手動でlog

CATEGORICAL_COLS = ["store_id_code", "product_id_code", "category_code"]

with mlflow.start_run(run_name="daily_sales_quantity_lgbm") as run:
    model = lgb.LGBMRegressor(
        objective="tweedie",
        tweedie_variance_power=1.3,
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=6,
        random_state=42,
        n_jobs=-1,
    )
    model.fit(X_train, y_train, categorical_feature=CATEGORICAL_COLS)

    y_pred = model.predict(X_test)
    rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
    mae = float(mean_absolute_error(y_test, y_pred))
    r2 = float(r2_score(y_test, y_pred))

    mlflow.log_metric("test_rmse", rmse)
    mlflow.log_metric("test_mae", mae)
    mlflow.log_metric("test_r2", r2)

    signature = infer_signature(X_train, model.predict(X_train))
    mlflow.lightgbm.log_model(
        model,
        artifact_path="model",
        signature=signature,
        input_example=X_train.iloc[:5],
    )

    run_id = run.info.run_id

print(f"run_id={run_id}, RMSE={rmse:.3f}, MAE={mae:.3f}, R2={r2:.3f}")

## Unity Catalog Model Registryへの登録

モデル名は `workspace.model.daily_sales_quantity_predictor` とする。テーブル層（bronze/silver/gold）で使われている `_NN_` の連番プレフィックスは、レイヤー間の処理順序を表すための命名規則であり、Model Registryのエントリには適用しない（モデルはテーブルのようなレイヤー処理順を持たず、Registry自体がバージョン管理を行うため）。

In [ ]:
model_name = "workspace.model.daily_sales_quantity_predictor"
model_uri = f"runs:/{run_id}/model"

registered_model = mlflow.register_model(model_uri=model_uri, name=model_name)
print(f"registered: {registered_model.name} v{registered_model.version}")

In [ ]:
client = MlflowClient()
client.set_registered_model_alias(
    name=model_name,
    alias="champion",
    version=registered_model.version,
)

In [ ]:
loaded_model = mlflow.pyfunc.load_model(f"models:/{model_name}@champion")

sample = X_test.iloc[:5]
preds = loaded_model.predict(sample)

result_df = pd.DataFrame({
    "sale_date": test_df["sale_date"].iloc[:5].values,
    "store_id": test_df["store_id"].iloc[:5].values,
    "product_id": test_df["product_id"].iloc[:5].values,
    "actual_quantity": y_test.iloc[:5].values,
    "predicted_quantity": preds,
})
display(result_df)

## 結果サマリ

- LightGBM（Tweedie目的関数）による日次売上数量予測（店舗×商品×日単位）
- 直近7日をテスト期間とした時系列分割でRMSE/MAE/R2を評価
- モデルは `workspace.model.daily_sales_quantity_predictor` としてUC Model Registryに登録し、`@champion` エイリアスを付与、`mlflow.pyfunc.load_model` での読み込みとpredictを確認済み

## 今後のTODO

- バッチ推論（スコアリング）notebookの追加検討
- ゼロインフレーション対応（2段階モデル / Tweedie損失など）の検討
- モデルエイリアス運用ルール（champion/challenger）の整備